# 02 — Cross-Validation of the Deterministic Engine

Runs the assessment engine over a grid of synthetic patients and checks
invariants that must hold for **every** input:

1. No recommended food/exercise ever carries an excluded tag (safety gate).
2. Risk labels match the cited thresholds at band edges.
3. The engine is deterministic: same input → identical output.

In [ ]:
import sys
from itertools import product
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.api.services import build_assessment
from src.db.models import Patient, Visit

def synthetic(sex, birth_year, glucose, sbp, dbp, hb, pregnant=False):
    patient = Patient(name="synthetic", sex=sex, birth_year=birth_year,
                      height_cm=162, pregnant=pregnant, conditions=[],
                      fitness_level="beginner")
    visit = Visit(weight_kg=68, biomarkers={
        "fasting_glucose": glucose, "systolic_bp": sbp,
        "diastolic_bp": dbp, "hemoglobin": hb})
    return patient, visit

In [ ]:
# Invariant 1+3: gate safety and determinism across a parameter grid
violations = 0
cases = 0
for sex, glucose, sbp, hb in product(
        ["male", "female"], [80, 110, 130], [115, 135, 150], [9.0, 13.0, 14.5]):
    patient, visit = synthetic(sex, 1990, glucose, sbp, 80, hb)
    result = build_assessment(patient, visit)
    again = build_assessment(patient, visit)
    assert result == again, "engine is not deterministic!"
    excluded = set(result["exclusions"]["food_tags"])
    for food in result["nutrition"]["top_foods"]:
        if excluded & set(food["tags"]):
            violations += 1
    cases += 1
print(f"{cases} grid cases, {violations} gate violations (must be 0)")

## Band-edge audit
The parametrized table in `tests/test_risk_scoring.py` is the authoritative
validation table; this cell re-runs it inline for the thesis record.

In [ ]:
import subprocess
print(subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_risk_scoring.py", "-v"],
    cwd=PROJECT_ROOT, capture_output=True, text=True).stdout[-3000:])